In [1]:
import numpy as np
from scipy.linalg import lu

In [ ]:
n = int(input("n = "))
t = int(input("t (eps = 10^-t) = "))
eps = 10 ** (-t)

# Generare matrice SPD: A = B B^T
np.random.seed(0)
B = np.random.rand(n, n)
A = B @ B.T
b = np.random.rand(n)


# LU folosind biblioteca
P, L_lib, U_lib = lu(Ainit)
xlib = np.linalg.solve(Ainit, b)

print("Descompunere LU (biblioteca):")
print("L =", L_lib)
print("U =", U_lib)
print("Solutia xlib =", xlib)

Descompunere LU (biblioteca):
L = [[ 1.          0.          0.          0.          0.        ]
 [ 0.87313294  1.          0.          0.          0.        ]
 [ 0.71690661 -0.41638046  1.          0.          0.        ]
 [ 0.98399178  0.91390902 -0.79682294  1.          0.        ]
 [ 0.89279265  0.16581043 -0.619071    0.81462945  1.        ]]
U = [[ 1.89249171  2.47959417  2.16857618  1.89111097  2.19083846]
 [ 0.         -0.27252364 -0.20385262 -0.29445147 -0.05069695]
 [ 0.          0.         -0.30484431  0.58535138 -0.39580064]
 [ 0.          0.          0.          0.07062076  0.00770845]
 [ 0.          0.          0.          0.         -0.00840129]]
Solutia xlib = [  77.00029406 -121.39859873  162.9267214    15.69743843 -103.87092299]


In [3]:
# Descompunere LDL^T (Choleski)
d = np.zeros(n)

for p in range(n):
    # calcul dp
    suma = 0.0
    for k in range(p):
        suma += d[k] * (A[p][k] ** 2)

    d[p] = A[p][p] - suma

    if abs(d[p]) < eps:
        raise ValueError("Matricea nu este pozitiv definita.")

    # calcul elemente L sub diagonala
    for i in range(p + 1, n):
        suma = 0.0
        for k in range(p):
            suma += d[k] * A[i][k] * A[p][k]

        A[i][p] = (A[i][p] - suma) / d[p]

In [4]:
# det(A) = det(L) det(D) det(L^T)
# det(L)=1, det(L^T)=1
# => det(A)=prod(di)
detA = np.prod(d)
print("Determinant A =", detA)

Determinant A = 9.32813199225323e-05


In [5]:
# Rezolvare Ax=b cu LDL^T
# L z = b (substitutie directa, diag=1)
z = np.zeros(n)
for i in range(n):
    suma = 0.0
    for j in range(i):
        suma += A[i][j] * z[j]
    z[i] = b[i] - suma

# D y = z
y = np.zeros(n)
for i in range(n):
    if abs(d[i]) < eps:
        raise ValueError("Impartire imposibila.")
    y[i] = z[i] / d[i]


# L^T x = y (substitutie inversa, diag=1)
xChol = np.zeros(n)
for i in reversed(range(n)):
    suma = 0.0
    for j in range(i + 1, n):
        suma += A[j][i] * xChol[j]
    xChol[i] = y[i] - suma

print("Solutia xChol =", xChol)

Solutia xChol = [  77.00029406 -121.39859873  162.9267214    15.69743843 -103.87092299]


In [ ]:
# Calcul manual Ainit * xChol
Ax = np.zeros(n)

for i in range(n):
    # element diagonal
    Ax[i] += A[i][i] * xChol[i]

    # partea superioara
    for j in range(i + 1, n):
        aij = A[i][j]   # partea superioara contine A initial
        Ax[i] += aij * xChol[j]
        Ax[j] += aij * xChol[i]   # folosim simetria

norm1 = np.linalg.norm(Ax - b, 2)
norm2 = np.linalg.norm(xChol - xlib, 2)

print("||Ainit xChol - b||2 =", norm1)
print("||xChol - xlib||2 =", norm2)

||Ainit xChol - b||2 = 6.142273138767745e-14
||xChol - xlib||2 = 1.3772727272469674e-11
